In [ ]:
import pandas as pd
import numpy as np
df = pd.read_parquet("../data/processed/cleaned_clv_data.parquet")

### The time period for the data used here is 1 year (2021)

In [421]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36086 entries, 0 to 36085
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   customer_id           36086 non-null  int64         
 1   transaction_count     36086 non-null  int64         
 2   promo_usage_count     36086 non-null  int64         
 3   quantity              36086 non-null  int64         
 4   avg_order_value       36086 non-null  float64       
 5   first_purchase_date   36086 non-null  datetime64[ns]
 6   last_purchase_date    36086 non-null  datetime64[ns]
 7   purchase_recency      36086 non-null  float64       
 8   customer_tenure       36086 non-null  float64       
 9   payment_method_count  36086 non-null  int64         
 10  future_spend          36086 non-null  float64       
 11  total_spent           36086 non-null  float64       
dtypes: datetime64[ns](2), float64(5), int64(5)
memory usage: 3.3 MB


In [422]:
df["total_spent"].sum()

25577540.15

__We made over $25.6 million in 2021__

In [423]:
# How much our various customer distribution made

df_sorted = df.sort_values('total_spent', ascending=False).reset_index(drop=True)

df_sorted['cum_revenue'] = df_sorted['total_spent'].cumsum()
df_sorted['cum_revenue_pct'] = (
    df_sorted['cum_revenue'] / df_sorted['total_spent'].sum()
) * 100

# Cumulative customer percentage
df_sorted['cum_customer_pct'] = (
    (df_sorted.index + 1) / len(df_sorted)
) * 100

print("Based on Total Spent alone:")
for p in [5, 10, 20, 30, 50]:
    revenue_share = df_sorted.loc[
        df_sorted['cum_customer_pct'] <= p, 
        'total_spent'
    ].sum() / df['total_spent'].sum() * 100
    
    print(f"Top {p}% customers generate {revenue_share:.2f}% of revenue")

Based on Total Spent alone:
Top 5% customers generate 46.30% of revenue
Top 10% customers generate 63.93% of revenue
Top 20% customers generate 81.55% of revenue
Top 30% customers generate 89.90% of revenue
Top 50% customers generate 96.87% of revenue


<hr/>

# RFM Segmentation

Assigning scores based on different values of RFM (e.g 5-5-5 means high recency, high frequency and total monetary value)

In [424]:
RFM = df[["customer_id", "purchase_recency", "transaction_count", "total_spent"]].copy()

## 1. Recency Score (1-5)

The lower the `purchase_recency`, the higher the recency score. (Top 20% lowest have a score of 5)

In [425]:
RFM["recency_score"] = pd.qcut(RFM["purchase_recency"], q=5, labels=[5, 4, 3, 2, 1])

recency_bins = RFM.groupby('recency_score', observed=True)['purchase_recency'].agg(['min', 'max']).sort_index(ascending=False)

print("Score Ranges:")
print(recency_bins)

Score Ranges:
                      min         max
recency_score                        
1              129.582811  364.948454
2               54.738876  129.577530
3               25.127685   54.735412
4                9.787500   25.126849
5                0.000191    9.786073


## 2. Frequency Score

Shows how often they make a purchase

In [426]:
RFM["transaction_count"].describe()

count    36086.000000
mean         6.793161
std          9.341450
min          1.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        160.000000
Name: transaction_count, dtype: float64

25% of customers only made one purchase in the time period

In [427]:
RFM['frequency_score'] = pd.qcut(RFM['transaction_count'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

frequency_bins = RFM.groupby('frequency_score', observed=True)['transaction_count'].agg(['min', 'max']).sort_index(ascending=False)

print("Score Ranges:")
print(frequency_bins)

Score Ranges:
                 min  max
frequency_score          
5                 10  160
4                  5   10
3                  2    5
2                  1    2
1                  1    1


## 3. Monetary Score

Customers are given scores based on how much they have spent

In [428]:
RFM["monetary_score"] = pd.qcut(RFM["total_spent"], q=5, labels=[1, 2, 3, 4, 5])

monetary_bins = RFM.groupby('monetary_score', observed=True)['total_spent'].agg(['min', 'max']).sort_index(ascending=False)

print("Score Ranges:")
print(monetary_bins)

Score Ranges:
                    min        max
monetary_score                    
5               819.485  52428.261
4               233.454    819.397
3                75.413    233.440
2                25.305     75.369
1                 1.635     25.303


<hr/>

## Segment Labelling

We'll come up with customer label groups based on the different RFM scores. Setting RFM values for segments is what really is responsible for separating our customers and personalizing retention efforts

In [429]:
RFM["recency_score"] = RFM["recency_score"].astype(int)
RFM["frequency_score"] = RFM["frequency_score"].astype(int)
RFM["monetary_score"] = RFM["monetary_score"].astype(int)

r, f, m = RFM["recency_score"], RFM["frequency_score"], RFM["monetary_score"]

RFM["segment_label"] = np.select(
    [
        (r >= 4) & (f >= 4) & (m == 5),     # Champions -> Focused retention efforts, upsells and cross sells
        (r >= 4) & (f >= 3) & (m >= 4),     # Loyal High Value -> Loyalty/referral programs
        (r <= 2) & (m >= 4),                # Lost High Value: Champions that have churned -> Win-back campaigns
        (r >= 4) & (f <= 2),                # New Buyers -> Post-purchase campaigns to keep them coming back
        (r >= 3) & (f >= 4) & (m >= 4),     # Potential Loyalists -> Incentives to keep them and increase how often they buy
        (r <= 2) & (f >= 4) & (m >= 4),     # At-Risk High Value: High value customers showing churn signals (there are non in this data)
        (r <= 2) & (f >= 3) & (m >= 3),     # Needs Attention: moderate spenders slowly churning
        (r >= 3) & (f >= 3) & (m >= 3),     # Engaged Mid-Value — active, consistent, moderate spend
        (r >= 3) & (f <= 2) & (m >= 3),     # Occasional Spenders — infrequent but decent value when they do buy
        (r <= 2) & (f <= 2) & (m <= 2),     # Low Value
    ],
    [
        "Champions",
        "Loyal High Value",
        "Lost High Value",
        "New Customers",
        "Potential Loyalists",
        "At-Risk High Value",
        "Needs Attention",
        "Engaged Mid-Value",
        "Occasional Spenders",
        "Low Value",
    ],
    default="Regulars"   
)


In [430]:
# Grouping segments by revenue and customers percentage
segment_revenue = RFM.groupby('segment_label')['total_spent'].sum()
segment_customers = RFM.groupby('segment_label')['customer_id'].count()

revenue_percentage = (segment_revenue / segment_revenue.sum()) * 100
customers_percentage = (segment_customers / segment_customers.sum()) * 100

revenue_summary = pd.DataFrame({
    'Total Revenue': segment_revenue,
    'Revenue(%)': revenue_percentage,
    'Customers(%)' : customers_percentage,
    'Customers': segment_customers,
}).sort_values(by='Revenue(%)', ascending=False).reset_index()

revenue_summary['Revenue(%)'] = revenue_summary['Revenue(%)'].round(2)
revenue_summary['Customers(%)'] = revenue_summary['Customers(%)'].round(2)


print(revenue_summary)

         segment_label  Total Revenue  Revenue(%)  Customers(%)  Customers
0            Champions   1.634955e+07       63.92         14.90       5377
1     Loyal High Value   2.527127e+06        9.88         12.36       4460
2  Potential Loyalists   2.223357e+06        8.69          5.31       1917
3      Lost High Value   2.010631e+06        7.86          4.99       1802
4    Engaged Mid-Value   1.233759e+06        4.82         13.69       4941
5             Regulars   3.709543e+05        1.45         16.31       5887
6  Occasional Spenders   3.587378e+05        1.40          1.66        600
7            Low Value   2.225062e+05        0.87         25.82       9318
8        New Customers   1.441382e+05        0.56          2.05        741
9      Needs Attention   1.367771e+05        0.53          2.89       1043


Now we have a comprehensive list of different customer groups based on how much they have spent. It's clear just how important __Champions__ category is, with only __14.9% of them making up 63.9%(about $16.3m)__ of the $25.6 million that was made in 2021 making each worth an average of __$3,040__